# Lab | Summarization evaluation using LangSmith

Compare base T5, CNN/DailyMail fine-tuned T5, and an OpenAI chat model on two held-out news datasets: CNN/DailyMail and XSum. Use five examples per dataset to keep the experiment small (30 summaries total). No capstone dataset was supplied, so XSum provides a second summarization style.

**Execution status:** Implementation and offline checks completed; live experiments require package installation, model downloads, and API keys. No scores or winning model are claimed before running.

Run cells from top to bottom. Install dependencies with `%pip install -r requirements.txt` in a notebook cell, then restart the kernel. Copy `.env.example` to `.env` and fill in credentials. Local T5 and embedding models require downloads and sufficient RAM; only the OpenAI generation calls incur provider API costs.

## LangSmith - LangChain evaluation

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

load_dotenv(Path.cwd() / ".env")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not LANGSMITH_API_KEY:
    raise RuntimeError("Set LANGSMITH_API_KEY in .env before running the experiments.")
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in .env to include the OpenAI comparison.")

In [ ]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "summarization-lab")
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY

In [ ]:
from langsmith import Client
client = Client(api_key=LANGSMITH_API_KEY)

### Create Dataset


In [ ]:
from datasets import load_dataset
from itertools import islice

MAX_NEWS = 5
SEED = 42
# Stream held-out data; shuffle a bounded buffer instead of downloading all splits.
DATASETS = {
    "cnn_dailymail": ("abisee/cnn_dailymail", "3.0.0", "article", "highlights"),
    "xsum": ("EdinburghNLP/xsum", None, "document", "summary"),
}

def load_examples(spec):
    repo, config, article_key, summary_key = spec
    stream = load_dataset(repo, config, split="test", streaming=True)
    stream = stream.shuffle(seed=SEED, buffer_size=1000)
    return [
        {"id": str(row.get("id", i)), "article": row[article_key],
         "highlights": row[summary_key]}
        for i, row in enumerate(islice(stream, MAX_NEWS))
    ]

samples = {name: load_examples(spec) for name, spec in DATASETS.items()}

In [ ]:
# Use the same bounded source text for all three models.
# T5's task prefix is added at generation time, not to the stored article.
from transformers import AutoTokenizer
source_tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-base")
MAX_SOURCE_TOKENS = 480
for rows in samples.values():
    for row in rows:
        ids = source_tokenizer.encode(row["article"], add_special_tokens=False)
        row["original_tokens"] = len(ids)
        row["article"] = source_tokenizer.decode(ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True)
        row["truncated"] = len(ids) > MAX_SOURCE_TOKENS

In [ ]:
{name: len(rows) for name, rows in samples.items()}

In [ ]:
samples["cnn_dailymail"][0]

In [ ]:
import pandas as pd
pd.DataFrame([
    {"dataset": name, "examples": len(rows),
     "truncated_articles": sum(row["truncated"] for row in rows)}
    for name, rows in samples.items()
])

Both datasets are normalized to `article`, `highlights`, and `id`. The article is the model input; highlights are the human reference. Each model sees the same truncated article. References remain unchanged, so missing facts caused by truncation can lower scores; inspect that limitation separately.

In [ ]:
print(samples["xsum"][0])

Now We have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith with this information.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input, which is the prompt passed to the model for evaluation, and an output, which should contain what we expect the model to return.

In [ ]:
from datetime import datetime, timezone
import uuid
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:8]

In [ ]:
dataset_names = {name: f"summarization-{name}-{RUN_ID}" for name in samples}

In [ ]:
datasets_by_name = {}
for name, rows in samples.items():
    dataset = client.create_dataset(
        dataset_name=dataset_names[name],
        description=f"Held-out {name}; seed={SEED}; n={MAX_NEWS}; source limit={MAX_SOURCE_TOKENS}",
    )
    client.create_examples(
        dataset_id=dataset.id,
        inputs=[{"article": row["article"]} for row in rows],
        outputs=[{"highlights": row["highlights"]} for row in rows],
        metadata=[{"source_id": row["id"], "truncated": row["truncated"],
                   "original_tokens": row["original_tokens"]} for row in rows],
    )
    datasets_by_name[name] = dataset
print(dataset_names)

Each LangSmith example now contains an article input and a human reference output. Re-running the dataset setup creates fresh datasets with unique names. Re-run only the evaluation cells to reuse the existing datasets.

### Load the two T5 models locally

Use Transformers directly because the original HuggingFaceHub integration is obsolete. Both models use deterministic decoding and identical generation limits. Models load sequentially to limit memory use.

In [ ]:
import gc
import torch
from transformers import AutoModelForSeq2SeqLM
from langsmith import traceable

MAX_NEW_TOKENS = 120

def make_t5_target(repo_id):
    tokenizer = AutoTokenizer.from_pretrained(repo_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(repo_id)
    model.eval()

    @traceable(name=repo_id, run_type="chain")
    def summarize(inputs):
        encoded = tokenizer("summarize: " + inputs["article"], return_tensors="pt",
                            truncation=True, max_length=512)
        with torch.inference_mode():
            output = model.generate(**encoded, max_new_tokens=MAX_NEW_TOKENS,
                                    do_sample=False, num_beams=4)
        return {"summary": tokenizer.decode(output[0], skip_special_tokens=True)}
    return summarize

In [ ]:
BASE_MODEL = "google-t5/t5-base"

In [ ]:
FINETUNED_MODEL = "flax-community/t5-base-cnn-dm"

## Define evaluators

Use cosine embedding distance (lower is better), ROUGE-L F1 (higher is better), and normalized edit distance (lower is better). Embeddings capture semantic similarity; ROUGE measures word overlap; edit distance is sensitive to phrasing. None establishes factual correctness. Manually inspect hallucinations, omissions, and preservation of names and numbers.

In [ ]:
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from rapidfuzz.distance import Levenshtein

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

In [ ]:
def embedding_distance(outputs, reference_outputs):
    vectors = embedder.encode(
        [outputs["summary"], reference_outputs["highlights"]], normalize_embeddings=True
    )
    distance = 1.0 - float(vectors[0] @ vectors[1])
    return {"key": "embedding_distance", "score": max(0.0, min(2.0, distance))}

def rouge_l(outputs, reference_outputs):
    score = rouge.score(reference_outputs["highlights"], outputs["summary"])["rougeL"].fmeasure
    return {"key": "rouge_l", "score": score}

def string_distance(outputs, reference_outputs):
    score = Levenshtein.normalized_distance(reference_outputs["highlights"], outputs["summary"])
    return {"key": "string_distance", "score": score}

evaluators = [embedding_distance, rouge_l, string_distance]

# Sanity checks: identical nonempty summaries should receive ideal scores.
example = "The council approved a new park."
assert embedding_distance({"summary": example}, {"highlights": example})["score"] < 1e-5
assert rouge_l({"summary": example}, {"highlights": example})["score"] == 1.0
assert string_distance({"summary": example}, {"highlights": example})["score"] == 0.0

### Run evaluations

Every model is evaluated on the same examples in both datasets. Concurrency is one to keep local memory and API usage bounded. Exceptions are not ignored: inspect errors before interpreting averages.

In [ ]:
experiments = {}

def evaluate_model(label, target):
    for name, dataset in datasets_by_name.items():
        experiments[(name, label)] = client.evaluate(
            target, data=dataset.name, evaluators=evaluators,
            experiment_prefix=f"{label}-{name}-{RUN_ID}",
            metadata={"model": label, "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS},
            max_concurrency=1,
        )

summarizer_base = make_t5_target(BASE_MODEL)
evaluate_model("t5-base", summarizer_base)
del summarizer_base
gc.collect()

In [ ]:
summarizer_finetuned = make_t5_target(FINETUNED_MODEL)
evaluate_model("t5-finetuned", summarizer_finetuned)
del summarizer_finetuned
gc.collect()

Open the experiment links printed by LangSmith to compare predictions, evaluator scores, latency, and errors for the two T5 models. Check both datasets separately: fine-tuning on CNN/DailyMail may not transfer equally to XSum.

### Compare an OpenAI model

This section makes ten paid generation calls with the default sample size. The model name is configurable in `.env`. Local embeddings avoid additional embedding API charges.

In [ ]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai

openai_client = wrap_openai(OpenAI(api_key=OPENAI_API_KEY))
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

@traceable(name="openai-summary", run_type="chain")
def openai_summarizer(inputs):
    response = openai_client.chat.completions.create(
        model=OPENAI_MODEL, temperature=0, max_tokens=MAX_NEW_TOKENS,
        messages=[
            {"role": "system", "content": "Summarize the news concisely. Preserve key facts and do not add unsupported information."},
            {"role": "user", "content": inputs["article"]},
        ],
    )
    summary = response.choices[0].message.content
    if not summary:
        raise ValueError("OpenAI returned an empty summary")
    return {"summary": summary}

In [ ]:
evaluate_model(OPENAI_MODEL, openai_summarizer)

### Results and comparison

The next cell exports per-example records and a grouped table. Compare quality alongside latency and failure counts. Token usage and available API cost information can be inspected in the OpenAI traces. Local inference still has hardware costs.

records = []
for (dataset_name, model_name), result in experiments.items():
    for row in result:
        run = row["run"]
        reference = row["example"]
        feedback = row["evaluation_results"]["results"]
        record = {
            "dataset": dataset_name, "model": model_name,
            "example_id": str(reference.id),
            "article": reference.inputs["article"],
            "reference": reference.outputs["highlights"],
            "summary": (run.outputs or {}).get("summary", ""),
            "error": str(run.error) if run.error else "",
            "latency_seconds": (run.end_time - run.start_time).total_seconds() if run.end_time else None,
            "evaluator_errors": sum(item.score is None for item in feedback),
        }
        record.update({item.key: item.score for item in feedback})
        records.append(record)

results = pd.DataFrame(records)
results["failed"] = results["error"].ne("") | results["summary"].eq("")
# Missing feedback must not silently look like a successful evaluation.
for metric in ["embedding_distance", "rouge_l", "string_distance"]:
    if metric not in results:
        results[metric] = float("nan")
comparison = results.groupby(["dataset", "model"]).agg(
    examples=("example_id", "count"), failures=("failed", "sum"),
    evaluator_errors=("evaluator_errors", "sum"),
    scored_examples=("embedding_distance", "count"),
    embedding_distance=("embedding_distance", "mean"),
    rouge_l=("rouge_l", "mean"), string_distance=("string_distance", "mean"),
    mean_latency_seconds=("latency_seconds", "mean"),
).reset_index()
results.to_csv("summarization_results.csv", index=False)
comparison.to_csv("summarization_comparison.csv", index=False)
display(comparison)
display(results[["dataset", "model", "reference", "summary", "error"]])

## What I learned

- Evaluation requires a stable input/output contract: all models return `summary`, and all references use `highlights`.
- A fair comparison holds examples, source truncation, and generation limits constant. Different tokenizers and prompts still prevent perfectly identical conditions.
- A model tuned on CNN/DailyMail may benefit on that dataset; XSum tests transfer to shorter, more abstractive summaries. This is a hypothesis until the experiments run.
- Semantic similarity, lexical overlap, and edit distance measure different properties. A fluent summary can still invent a number or omit a key fact; human review is necessary.
- The embedding model also has an input-length limit, so long references may be truncated during scoring.
- Five examples per dataset are a smoke test, not enough to establish a reliable winner. A stronger study uses larger representative samples, repeated runs, and uncertainty estimates.
- Quality, latency, errors, and API cost all matter. Temperature zero reduces randomness but does not guarantee reproducibility across provider/model updates.

**Conclusion:** Live results are pending. The original claim that OpenAI wins cannot be carried over without measurement. After execution, use the exported comparison and inspect per-example factual errors before choosing a model.

Implementation reference: [LangSmith evaluation quickstart](https://docs.langchain.com/langsmith/evaluation-quickstart) and [custom evaluators](https://docs.langchain.com/langsmith/code-evaluator-sdk).
